In [6]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.models import load_model


In [2]:
word_index = imdb.get_word_index()
reverse_word_index = {value:key for key, value in word_index.items()} 

In [7]:
from tensorflow.keras import Input

model = Sequential([
    Input(shape=(500,)),
    Embedding(10000,128),
    SimpleRNN(128),
    Dense(1,activation='sigmoid')
])

In [8]:
model.save("simple_rnn_imdb.h5")

In [12]:
from tensorflow.keras.models import load_model

model = load_model("simple_rnn_imdb.h5")
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
model.get_weights()

[array([[ 0.04288689,  0.01011226, -0.04616509, ...,  0.01550695,
         -0.02846082,  0.01427051],
        [-0.04108772,  0.01444317, -0.02162085, ..., -0.02766421,
          0.04524187,  0.03145966],
        [ 0.03713853,  0.02366097, -0.00876731, ...,  0.02448056,
          0.00520339,  0.03750286],
        ...,
        [ 0.02752296, -0.04918001, -0.01116649, ...,  0.02797042,
         -0.03268633, -0.02018527],
        [ 0.04003484,  0.00752486, -0.02834886, ..., -0.02641878,
         -0.04821997,  0.01665851],
        [-0.038973  , -0.02685864, -0.02859022, ...,  0.01876925,
         -0.04088478, -0.00850549]], shape=(10000, 128), dtype=float32),
 array([[-0.15091226,  0.0292872 , -0.1473283 , ..., -0.01592374,
          0.0312053 ,  0.08822814],
        [ 0.06539255, -0.0118209 ,  0.10061876, ...,  0.06499405,
          0.13017042,  0.14894639],
        [-0.03318993,  0.03058359, -0.1059352 , ...,  0.03428456,
          0.12312151,  0.10214286],
        ...,
        [-0.0728937

In [15]:
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

In [16]:
def preprocess_text(text):
    # Tokenize the text into words
    words = text.lower().split()
    
    # Convert words to their corresponding indices in the word index
    encoded_review = [word_index.get(word, 2) + 3 for word in words]  # 2 is for unknown words
    
    # Pad the sequence to ensure it has a length of 500
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    
    return padded_review

In [17]:
##prediction func
def predict_sentiment(review):
    preprocessed_input = preprocess_text(review)
    prediction = model.predict(preprocessed_input)
    sentiment = "Positive" if prediction[0][0] > 0.5 else "Negative"
    return sentiment, prediction[0][0]


In [18]:
##user input and pred
example_review = "This movie was fantastic! I loved it."
sentiment, score = predict_sentiment(example_review)
print(f"Review: {example_review}\nPredicted Sentiment: {sentiment} (Score: {score})")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 304ms/step
Review: This movie was fantastic! I loved it.
Predicted Sentiment: Positive (Score: 0.5174918174743652)
